# Named Entit recognition

In [1]:
import spacy
from nltk import sent_tokenize

In [2]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 5.1 MB/s  0:01:44 eta 0:00:010:00:03
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


# Load model

In [3]:
def load_model():
    nlp = spacy.load("en_core_web_trf")
    return nlp

In [4]:
nlp_model = load_model()

/home/keshtech/miniconda3/envs/nlp_series/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/keshtech/miniconda3/envs/nlp_series/lib/python3.10/site-packages/thinc/shims/pytorch.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via 

# load Dataset

In [5]:
import os 
import sys
import pathlib
folder_path = pathlib.Path().parent.resolve()
sys.path.append(os.path.join(folder_path, '../'))
from utils import load_subtitles_dataset

In [6]:
dataset_path = "../data/Subtitles/"
df = load_subtitles_dataset(dataset_path)

In [7]:
df.head()

,episode,script
0,118,Connecting old words that have been used up\n ...
1,112,Connecting old words that have been used up\n ...
2,24,"C'mon!\n Running like a fugitive,\n Being chas..."
3,148,"Hey boys, listen up!\n Humans aren’t such a bi..."
4,123,Connecting old words that have been used up\n ...


In [8]:
sample_script = df.iloc[0]['script']
sample_script

"Connecting old words that have been used up\n I just want to convey my simple thoughts\n That day's faint wind blows away\n at the castle ruins of yesterday\n I'll laugh with you and live for the moment\n Still that incident will continue to trouble you, right?\n That's why it must be good-bye\n Leaving nothing behind, let us share what is left\n In time, we'll realize that it was everything\n And sadness will touch our cheeks, creating a river of tears\n Trembling thoughts swept into a vortex and melting away\n Hurry!\n Right…\n We’re close!\n There are two enemies left…\n There are three of us… lt’s three against two.\n We might be able to do it with one more contact if we do it well.\n All right! All right! Let’s do it!\n No! It’s four against two!\n Okay, okay! I’m sorry.\n Counting Akamaru, it’s four against two!\n Let’s finish it quickly with our new technique next!\n Say, say. If necessary, I’ll do an awesome Shadow Clone,\n so it’ll be okay!\n That’s reassuring…\n But my strat

In [9]:
sentences = sent_tokenize(sample_script)

In [10]:
sentences = sentences[60:90]

In [11]:
sentence = ".".join(sentences)

In [12]:
sentence

'Lord Orochimaru…\n Several bodies have been prepared for you!.And…you could even use me, if necessary…\n It has to be Sasuke…\n How long do you think I’ve waited?.It’s a matter of urgency!.The Third Hokage’s Jutsu…\n His curse was stronger than expected!.Take this!.Sealing Jutsu!.Reaper Death Seal!.Wh-What’s this sensation?.Seal!.My arm…\n If you don’t “reanimate” right now, you’ll lose everything!.Is that what you want?!.Lose everything…\n I’ve selected and prepared proficient bodies for you..Please leave it to me!.Also…\n I’ll handle the recovery of Sasuke as well..Shall I use Kimimaro?.Curse you …\n Third Hokage!.Release!.Listen well, everyone!.I shall release you from this prison…\n However, it’ll only be one of you!.Now all that’s left is to wake up Kimimaro, huh?.Just as I thought…\n I wasn’t summoned this time, either..Dr. Kabuto, I…\n Take care of your body..Your role is now finished..It’s not Sound Ninja Five now, Kimimaro…\n I…still haven’t…\n Your replacement has already be

# Run Model

In [13]:
doc = nlp_model(sentence)

In [14]:
doc.ents

(Orochimaru,
 Sasuke,
 Third,
 Sasuke,
 Third,
 one,
 Kimimaro,
 Kabuto,
 Kimimaro,
 Kabuto)

In [15]:
for entity in doc.ents:
    print(entity, entity.label_)

Orochimaru PERSON
Sasuke PERSON
Third ORDINAL
Sasuke PERSON
Third ORDINAL
one CARDINAL
Kimimaro PERSON
Kabuto PERSON
Kimimaro PERSON
Kabuto PERSON


In [16]:
def get_ners_inference(script):
    script_sentences = sent_tokenize(script)

    ner_output = []

    for sentence in script_sentences:
        doc = nlp_model(sentence)
        ners = set()
        for entity in doc.ents:
            if entity.label_ =="PERSON":
                full_name = entity.text
                first_name = entity.text.split(" ")[0]
                first_name = first_name.strip()
                ners.add(first_name)
        ner_output.append(ners)

    return ner_output

In [17]:
df = df.head(10)

In [18]:
df

,episode,script
0,118,Connecting old words that have been used up\n ...
1,112,Connecting old words that have been used up\n ...
2,24,"C'mon!\n Running like a fugitive,\n Being chas..."
3,148,"Hey boys, listen up!\n Humans aren’t such a bi..."
4,123,Connecting old words that have been used up\n ...
5,121,Connecting old words that have been used up\n ...
6,180,"Rock away your existence,\n Shouting that you ..."
7,160,I want to try and gather the unrestrained wind...
8,168,I want to try and gather the unrestrained wind...
9,190,"Rock away your existence,\n Shouting that you ..."


In [19]:
df['ners'] = df['script'].apply(get_ners_inference)

In [20]:
df

,episode,script,ners
0,118,Connecting old words that have been used up\n ...,"[{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {..."
1,112,Connecting old words that have been used up\n ...,"[{}, {}, {}, {}, {}, {Sasuke}, {Naruto}, {}, {..."
2,24,"C'mon!\n Running like a fugitive,\n Being chas...","[{}, {}, {}, {Ibiki}, {}, {}, {}, {}, {}, {}, ..."
3,148,"Hey boys, listen up!\n Humans aren’t such a bi...","[{}, {}, {}, {}, {}, {Tsunade}, {}, {Hinata, K..."
4,123,Connecting old words that have been used up\n ...,"[{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {..."
5,121,Connecting old words that have been used up\n ...,"[{}, {}, {}, {}, {}, {}, {}, {Akamaru}, {}, {}..."
6,180,"Rock away your existence,\n Shouting that you ...","[{}, {}, {}, {}, {}, {Natsuhiboshi}, {}, {}, {..."
7,160,I want to try and gather the unrestrained wind...,"[{}, {}, {}, {}, {}, {Tokichi}, {}, {Hinata}, ..."
8,168,I want to try and gather the unrestrained wind...,"[{}, {}, {}, {}, {}, {}, {}, {Ramen}, {Ramen},..."
9,190,"Rock away your existence,\n Shouting that you ...","[{}, {}, {}, {}, {}, {}, {}, {Naruto}, {}, {},..."


In [23]:
dictionary = {"Sasuke"}
dict_list = list(dictionary)
dict_list

['Sasuke']

# Character Network

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

In [ ]:
def generate_character_network(df):

    windows=10
    entity_relationship = []

    for row in df['ners']:
        previous_entities_in_window = []

        for sentence in row:
            previous_entities_in_window.append(list(sentence))
            previous_entities_in_window = previous_entities_in_window[-windows:]

            # Flatten 2D List into 1D List
            previous_entities_flattened = sum(previous_entities_in_window, [])

            for entity in sentence:
                for entity_in_window in previous_entities_flattened:
                    if entity != entity_in_window:
                        entity_relationship.append(sorted([entity, entity_in_window]))
    
    relationship_df = pd.DataFrame({'value': entity_relationship})
    relationship_df['source'] = relationship_df['value'].apply(lambda x: x[0])
    relationship_df['target'] = relationship_df['value'].apply(lambda x: x[1])
    relationship_df = relationship_df.groupby(['source', 'target']).count().reset_index()
    relationship_df = relationship_df.sort_values('value', ascending=False)

    return relationship_df

In [25]:
relationship_df = generate_character_network(df)

In [26]:
relationship_df

,source,target,value
170,Hinata,Naruto,98
287,Naruto,Sasuke,32
94,Choji,Naruto,21
319,Orochimaru,Sasuke,20
88,Choji,Hinata,19
...,...,...,...
253,Lee,Naruto,1
114,Chunin,Naruto,1
113,Chunin,Lee,1
257,Lee,Roger,1


In [27]:
relationship_df = relationship_df.sort_values('value', ascending=False)
relationship_df = relationship_df.head(200)

In [28]:
G = nx.from_pandas_edgelist(
    relationship_df, 
    source='source', 
    target='target', 
    edge_attr='value',
    create_using=nx.Graph()
)

net = Network(notebook=True, width="1000px", height="700px", bgcolor="#222222", font_color="white", cdn_resources="remote")
node_degree = dict(G.degree)

nx.set_node_attributes(G, node_degree, 'size')
net.from_nx(G)
net.show("naruto.html")


naruto.html
